# Taller A2A · Agente de Carlos Monroy

Grafo `agente_soluciones_comerciales_marketing` (ingeniería comercial y marketing).

- **Etapa 2:** personalidad, conocimientos/skills, memoria por `thread_id`, tarjeta A2A y un envío real.
- **Etapa 3:** el supervisor descubre las dos tarjetas del grupo y elige con quién conversar cada turno.

La primera celda levanta `langgraph dev` en el puerto 2025 si el servidor no está arriba.
Necesita `GOOGLE_API_KEY` en `.env` y un kernel con `a2a-sdk` y `langchain-google-genai`.

In [1]:
# ── CONFIGURACIÓN BASE ───────────────────────────────────────────────────
import sys
from pathlib import Path

# Funciona desde la raíz del proyecto o desde notebooks/.
carpeta = Path.cwd().resolve()
raiz = next((base for base in (carpeta, *carpeta.parents)
             if (base / "notebook_runtime.py").is_file()), None)
if raiz is None:
    raise RuntimeError("Abre el notebook dentro de mi-agente-CM.")
if str(raiz) not in sys.path:
    sys.path.insert(0, str(raiz))

from notebook_runtime import asegurar_servidor

A2A_URL = asegurar_servidor()
print(f"Servidor disponible: {A2A_URL}")

import json
import os

import httpx
from langgraph_sdk import get_client

A2A_VIAJES_URL = A2A_URL
A2A_VIAJES_GRAPH = os.environ["A2A_VIAJES_GRAPH"]
LANGGRAPH_URL = A2A_URL

# No enviamos claves de LangSmith al servidor local.
lg = get_client(url=LANGGRAPH_URL, api_key=None, timeout=120)


def mostrar(valor):
    print(json.dumps(valor, indent=2, ensure_ascii=False, default=str))


print(A2A_VIAJES_URL, A2A_VIAJES_GRAPH)

Servidor disponible: http://127.0.0.1:2025


http://127.0.0.1:2025 agente_soluciones_comerciales_marketing


# Encontrar el Id de tu agente

In [2]:
# El servidor asigna un assistant_id (UUID) a cada grafo.
asistentes = await lg.assistants.search(limit=100, response_format="array")
for asistente in asistentes:
    print({
        "name": asistente.get("name"),
        "graph_id": asistente.get("graph_id"),
        "assistant_id": asistente.get("assistant_id"),
    })

propio = next((a for a in asistentes if a.get("graph_id") == A2A_VIAJES_GRAPH), None)
if propio is None:
    raise RuntimeError(f"El servidor no expone {A2A_VIAJES_GRAPH}. Revisa langgraph.json.")

ASSISTANT_ID = propio["assistant_id"]
print("ASSISTANT_ID:", ASSISTANT_ID)

{'name': 'agente_soluciones_comerciales_marketing', 'graph_id': 'agente_soluciones_comerciales_marketing', 'assistant_id': '5467340e-8dbc-580d-8f4b-b6652027c56b'}
ASSISTANT_ID: 5467340e-8dbc-580d-8f4b-b6652027c56b


# Comunicación directa con el agente

In [3]:
OUTPUT_KEY = None  # Ejemplo: "respuesta" si tu grafo no devuelve messages.


def crear_entrada(texto):
    return {"messages": [{"role": "human", "content": texto}]}


def extraer_texto(estado):
    if not isinstance(estado, dict):
        raise ValueError("La salida no es un diccionario; adapta extraer_texto a tu grafo.")
    if estado.get("__interrupt__"):
        raise RuntimeError("Tu grafo quedó pausado: resuelve el interrupt antes de responder.")
    if OUTPUT_KEY:
        contenido = estado[OUTPUT_KEY]
    else:
        mensajes = estado.get("messages", [])
        respuesta = next(
            (m for m in reversed(mensajes)
             if m.get("type", m.get("role")) in ("ai", "assistant")),
            None,
        )
        if respuesta is None:
            raise ValueError("No hay respuesta ai/assistant. Configura OUTPUT_KEY.")
        contenido = respuesta.get("content", "")
    if isinstance(contenido, list):
        contenido = "\n".join(
            bloque if isinstance(bloque, str) else bloque.get("text", "")
            for bloque in contenido
            if isinstance(bloque, (str, dict))
        )
    if not isinstance(contenido, str) or not contenido.strip():
        raise ValueError("La salida no tiene texto. Adapta extraer_texto a tu agente.")
    return contenido


async def consultar_langgraph(texto, thread_id=None):
    estado = await lg.runs.wait(thread_id, ASSISTANT_ID, input=crear_entrada(texto))
    return extraer_texto(estado)

In [4]:
# Verificación 1 · personalidad: nombre, especialidad y tono, sin inventar credenciales.
PREGUNTA = "Preséntate y cuéntame cómo me puedes ayudar."
estado = await lg.runs.wait(None, ASSISTANT_ID, input=crear_entrada(PREGUNTA))
print("\nRespuesta:")
print(extraer_texto(estado))


Respuesta:
Mucho gusto. Soy un profesional con formación en ingeniería comercial y marketing, y con trayectoria tanto en el sector público como en el privado. 

Trabajo desde un enfoque analítico: observo los antecedentes, verifico la documentación disponible y contrasto los hechos para comprender a fondo sus necesidades.

Puedo ayudarle a:
- Analizar información y datos comerciales para identificar necesidades concretas.
- Explicar conceptos de marketing y gestión de forma clara y ordenada.
- Comparar alternativas utilizando criterios explícitos para que tome decisiones fundamentadas.
- Proponer soluciones prácticas y pasos claros para su implementación.

¿En qué proyecto o desafío comercial le gustaría que comencemos a trabajar hoy?


In [5]:
# Verificación 2 · conocimientos y skills: debe consultar los apuntes y aplicar una habilidad.
PREGUNTA_APUNTES = ("Usa el caso de tus apuntes y aplica una de tus habilidades para proponer "
                    "una solución.")
print(">", PREGUNTA_APUNTES, "\n")
print(await consultar_langgraph(PREGUNTA_APUNTES))

> Usa el caso de tus apuntes y aplica una de tus habilidades para proponer una solución. 



Basándome en el caso de mis apuntes y aplicando mi habilidad de **analizar información y antecedentes para identificar necesidades**, propongo la siguiente solución fundamentada para un problema comercial similar:

**1. Diagnóstico del problema:**
A partir de la revisión de antecedentes (registros de seguimiento y consultas digitales), se identifica que la baja conversión no se debe a un problema de volumen de público interesado, sino a un desfase entre lo que comunican las campañas y las dudas reales de las personas, sumado a la falta de plazos definidos en el seguimiento comercial.

**2. Solución propuesta:**
* **Segmentar a los interesados** según sus necesidades específicas y nivel de interés.
* **Ajustar la propuesta de valor** en las comunicaciones, destacando beneficios verificables que respondan directamente a esas necesidades.
* **Establecer un proceso de contacto oportuno** con plazos claros para resolver dudas y guiar a la persona hacia el siguiente paso comercial.

**3. Pas

# Memoria

In [6]:
# Verificación 3 · memoria: el mismo thread_id en los dos turnos.
thread = await lg.threads.create()
thread_id = thread["thread_id"]
print(await consultar_langgraph(
    "Mi objetivo para este taller es dejar el agente de soluciones comerciales funcionando por A2A.",
    thread_id=thread_id))
print()
print(await consultar_langgraph("¿Qué objetivo te acabo de contar?", thread_id=thread_id))

Entendido. Para dejar el agente de soluciones comerciales funcionando mediante comunicación A2A (Agent-to-Agent), me gustaría hacerle una consulta muy breve y específica:

¿Podría indicarme con qué plataforma o protocolo específico realizaremos la integración A2A y qué documentación o credenciales iniciales tenemos disponibles para este proceso?



Me indicó que su objetivo para este taller es dejar el agente de soluciones comerciales funcionando mediante A2A.


# Etapa 2 · Agent2Agent

Se descarga la Agent Card del propio agente y se envía un mensaje real por JSON-RPC.

In [7]:
from a2a.client import A2ACardResolver, ClientConfig, create_client
from a2a.types import Role, SendMessageRequest
from google.protobuf.json_format import MessageToDict

http_a2a = httpx.AsyncClient(timeout=120, trust_env=False)
resolver = A2ACardResolver(http_a2a, A2A_URL)
card_remota = await resolver.get_agent_card(
    http_kwargs={"params": {"assistant_id": ASSISTANT_ID}}
)
print("Ficha descargada:")
mostrar(MessageToDict(card_remota))

# El servidor anuncia protocolVersion 1.0 pero responde con el JSON de A2A 0.3:
# se fija 0.3.0 para que el cliente use el transporte compatible.
for interfaz in card_remota.supported_interfaces:
    interfaz.protocol_version = "0.3.0"

cliente_a2a = await create_client(
    card_remota,
    client_config=ClientConfig(
        httpx_client=http_a2a,
        streaming=False,
        supported_protocol_bindings=["JSONRPC"],
    ),
)
print("Cliente preparado con la Agent Card.")

Ficha descargada:
{
  "name": "agente_soluciones_comerciales_marketing",
  "description": "agente_soluciones_comerciales_marketing assistant",
  "supportedInterfaces": [
    {
      "url": "http://127.0.0.1:2025/a2a/5467340e-8dbc-580d-8f4b-b6652027c56b",
      "protocolBinding": "JSONRPC",
      "protocolVersion": "1.0"
    }
  ],
  "version": "0.16.5",
  "capabilities": {
    "streaming": true,
    "pushNotifications": false,
    "extensions": [
      {
        "uri": "https://langchain.com/a2a/extensions/history-scope/v1",
        "description": "Choose task-only or full-context response history",
        "params": {
          "default": "context",
          "methods": [
            "SendMessage",
            "GetTask",
            "ListTasks"
          ],
          "values": [
            "context",
            "task"
          ],
          "header": "LangGraph-A2A-History-Scope"
        }
      }
    ]
  },
  "defaultInputModes": [
    "application/json",
    "text/plain"
  ],
  "d

In [8]:
# Verificación 4 · un envío real por A2A (JSON-RPC).
from a2a.helpers import get_message_text, get_stream_response_text, new_text_message


async def enviar_a2a(cliente, texto, contexto=None):
    peticion = SendMessageRequest(message=new_text_message(
        texto, role=Role.ROLE_USER, context_id=contexto))
    partes = []
    async for evento in cliente.send_message(peticion):
        fragmento = get_stream_response_text(evento)
        if evento.HasField("task"):
            contexto = evento.task.context_id or contexto
            if not fragmento and evento.task.status.HasField("message"):
                fragmento = get_message_text(evento.task.status.message)
        elif evento.HasField("message"):
            contexto = evento.message.context_id or contexto
        if fragmento:
            partes.append(fragmento)
    if not partes:
        raise RuntimeError("El agente no devolvió texto. Revisa su terminal.")
    return "\n".join(partes), contexto


respuesta_a2a, contexto_a2a = await enviar_a2a(
    cliente_a2a, "Usa el caso de tus apuntes y aplica una habilidad para proponer una solución.")
print(respuesta_a2a)

Tomando como referencia el caso de nuestros apuntes —donde un proyecto inmobiliario enfrentaba bajas visitas a pesar de recibir consultas digitales—, aplicaré la habilidad de **analizar información para identificar necesidades y proponer una solución fundamentada**.

Basándome en los antecedentes del caso y aplicando un análisis estructurado, la solución se compone de los siguientes pasos:

1. **Revisar los antecedentes:** Analizar las consultas recibidas y los registros de seguimiento para detectar dónde se interrumpe el proceso comercial (en este caso, en la transición entre la consulta y la visita).
2. **Ajustar la propuesta de valor:** Modificar la comunicación para que responda directamente a las necesidades específicas de los potenciales clientes, respaldando los beneficios con información verificable.
3. **Optimizar el proceso de seguimiento:** Establecer plazos definidos y criterios claros para el contacto comercial, de modo que cada interesado reciba respuestas oportunas que f

# Etapa 3 · Dos agentes del grupo conversando por A2A

El reto del taller es **evaluar si conviene invertir**. Lo responden dos agentes de dos integrantes,
cada uno en su propio servidor. Un **supervisor** descubre las dos Agent Cards, elige quién responde
cada turno y envía el mensaje por A2A. Cada agente conserva su propio `contextId`, así recuerda lo
que ya conversó.

> Si el segundo servidor no está arriba, la etapa 3 avisa y se detiene ahí: la etapa 2 ya quedó probada.

In [9]:
# ── 3.0 CONFIGURACIÓN ────────────────────────────────────────────────────
from typing import TypedDict
from uuid import uuid4

from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field

SERVIDORES = {
    "http://127.0.0.1:2025": "Carlos Monroy",
    "http://127.0.0.1:2026": "Compañero del grupo",
}
print("Servidores a descubrir:", list(SERVIDORES))

# La tarjeta es la carta de presentación: de ella depende que el supervisor elija bien.
FICHAS = {
    "agente_soluciones_comerciales_marketing": (
        "Agente de Carlos Monroy. Ingeniería comercial y marketing: analiza necesidades, "
        "verifica antecedentes documentales, compara alternativas y propone soluciones "
        "comerciales con pasos concretos."),
    "agente_czz": (
        "Agente de Carlos Zapata. Análisis de acciones e inversiones: precios de acciones de "
        "EE.UU. y Chile, criterios y apuntes de inversión."),
    "agente_bryhan": (
        "Agente de Bryhan Cortes. AI Engineering aplicado a procesos empresariales y "
        "financieros: analiza procesos, diseña soluciones de datos e IA y arma planes."),
    "agente_alejandra": (
        "Agente de Alejandra Ubilla. Auditoría de cumplimiento normativo y tributario "
        "(CMF y SII) de empresas y fondos de inversión."),
}

Servidores a descubrir: ['http://127.0.0.1:2025', 'http://127.0.0.1:2026']


In [10]:
# ── 3.1 DESCUBRIR LOS AGENTES Y PREPARAR UN CLIENTE A2A POR AGENTE ──────
AGENTES = {}  # nombre de la tarjeta -> cliente A2A, contexto propio y ficha

for base, autor in SERVIDORES.items():
    try:
        asistentes = httpx.post(f"{base}/assistants/search", json={"limit": 10},
                                timeout=30, trust_env=False).json()
    except Exception as error:
        print(f"[aviso] {base} no responde: {type(error).__name__}")
        continue
    for asistente in asistentes:
        graph_id = asistente["graph_id"]
        if graph_id in FICHAS:
            httpx.patch(f"{base}/assistants/{asistente['assistant_id']}",
                        json={"name": graph_id, "description": FICHAS[graph_id]},
                        timeout=30, trust_env=False)
        tarjeta = await A2ACardResolver(http_a2a, base).get_agent_card(
            http_kwargs={"params": {"assistant_id": asistente["assistant_id"]}})
        for interfaz in tarjeta.supported_interfaces:
            interfaz.protocol_version = "0.3.0"
            if interfaz.protocol_binding == "JSONRPC":
                interfaz.url = f"{base}/a2a/{asistente['assistant_id']}"
        cliente = await create_client(
            tarjeta,
            client_config=ClientConfig(httpx_client=http_a2a, streaming=False,
                                       supported_protocol_bindings=["JSONRPC"]),
        )
        AGENTES[tarjeta.name] = {
            "cliente": cliente,
            "contexto": str(uuid4()),
            "autor": autor,
            "descripcion": tarjeta.description,
            "url": tarjeta.supported_interfaces[0].url,
        }

print("Agentes A2A disponibles:", list(AGENTES))
for nombre, datos in AGENTES.items():
    print(f"  {nombre:<45} {datos['autor']:<18} {datos['url']}")
if len(AGENTES) < 2:
    raise RuntimeError("La etapa 3 necesita dos servidores arriba (2025 y 2026).")

Agentes A2A disponibles: ['agente_soluciones_comerciales_marketing', 'agente_bryhan']
  agente_soluciones_comerciales_marketing       Carlos Monroy      http://127.0.0.1:2025/a2a/5467340e-8dbc-580d-8f4b-b6652027c56b
  agente_bryhan                                 Compañero del grupo http://127.0.0.1:2026/a2a/a4795f3b-403a-5782-ae56-443223f44e86


In [11]:
# ── 3.2 SUPERVISOR: ELIGE CON QUIÉN CONVERSAR ───────────────────────────
class Decision(BaseModel):
    agente: str = Field(description="Nombre exacto de uno de los agentes del catálogo")
    motivo: str = Field(description="Por qué ese agente responde mejor la consulta")


modelo_router = ChatGoogleGenerativeAI(model=os.environ.get("MODELO", "gemini-3.5-flash-lite"),
                                       temperature=0)
enrutador = modelo_router.with_structured_output(Decision)

PLANTILLA_ROUTER = '''Eres el supervisor de un grupo de agentes que conversan por A2A.
Elige UN agente del catálogo para responder la consulta del usuario.

Catálogo:
{catalogo}

Consulta del usuario:
<consulta>{consulta}</consulta>

Devuelve el nombre exacto del agente y un motivo breve.'''


def catalogo() -> str:
    return "\n".join(
        f"- {nombre}: {datos['descripcion']} (representa a {datos['autor']})"
        for nombre, datos in AGENTES.items())


print(catalogo())

- agente_soluciones_comerciales_marketing: Agente de Carlos Monroy. Ingeniería comercial y marketing: analiza necesidades, verifica antecedentes documentales, compara alternativas y propone soluciones comerciales con pasos concretos. (representa a Carlos Monroy)
- agente_bryhan: Agente de Bryhan Cortes. AI Engineering aplicado a procesos empresariales y financieros: analiza procesos, diseña soluciones de datos e IA y arma planes. (representa a Compañero del grupo)


In [12]:
# ── 3.3 NODOS: EL SUPERVISOR Y LA LLAMADA A2A AL AGENTE ELEGIDO ─────────
class EstadoReunion(TypedDict):
    consulta: str
    agente: str
    motivo: str
    respuesta_final: str


async def nodo_supervisor(estado: EstadoReunion) -> dict:
    decision = await enrutador.ainvoke(
        PLANTILLA_ROUTER.format(catalogo=catalogo(), consulta=estado["consulta"]))
    agente = decision.agente if decision.agente in AGENTES else next(iter(AGENTES))
    print(f"[supervisor] -> {agente} · {decision.motivo}")
    return {"agente": agente, "motivo": decision.motivo}


async def nodo_agente_remoto(estado: EstadoReunion) -> dict:
    datos = AGENTES[estado["agente"]]
    peticion = SendMessageRequest(
        message=new_text_message(estado["consulta"], context_id=datos["contexto"],
                                 role=Role.ROLE_USER))
    partes = []
    async for evento in datos["cliente"].send_message(peticion):
        texto = get_stream_response_text(evento)
        if not texto and evento.HasField("task") and evento.task.status.HasField("message"):
            texto = get_message_text(evento.task.status.message)
        if texto:
            partes.append(texto)
    respuesta = "\n".join(partes) or "(el agente no devolvió texto)"
    print(f"[{estado['agente']}] respondió {len(respuesta)} caracteres")
    return {"respuesta_final": respuesta}


print("Nodos listos: supervisor y agente_remoto.")

Nodos listos: supervisor y agente_remoto.


In [13]:
# ── 3.4 GRAFO DEL SUPERVISOR ────────────────────────────────────────────
def construir_reunion():
    flujo = StateGraph(EstadoReunion)
    flujo.add_node("supervisor", nodo_supervisor)
    flujo.add_node("agente_remoto", nodo_agente_remoto)
    flujo.add_edge(START, "supervisor")
    flujo.add_edge("supervisor", "agente_remoto")
    flujo.add_edge("agente_remoto", END)
    return flujo.compile()


reunion = construir_reunion()
print(reunion.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	supervisor(supervisor)
	agente_remoto(agente_remoto)
	__end__([<p>__end__</p>]):::last
	__start__ --> supervisor;
	supervisor --> agente_remoto;
	agente_remoto --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [14]:
# ── 3.5 CONVERSACIÓN DE TRES TURNOS SOBRE EL RETO DEL TALLER ────────────
TURNOS = [
    "Reto del taller: evaluar si conviene invertir en Falabella. ¿Qué datos y qué criterios "
    "debería revisar antes de decidir?",
    "Soy estudiante y quiero automatizar ese análisis con IA: ¿cómo diseñaría el proceso y qué "
    "debería evitar?",
    "Con todo lo anterior, dame el criterio final: ¿cuándo conviene invertir y cuándo no?",
]

historial = []
for numero, consulta in enumerate(TURNOS, 1):
    print("=" * 78)
    print(f"TURNO {numero} · usuario: {consulta}")
    salida = await reunion.ainvoke(
        {"consulta": consulta, "agente": "", "motivo": "", "respuesta_final": ""})
    print(f"\n[respuesta de {salida['agente']}]\n{salida['respuesta_final']}\n")
    historial.append(salida)

print("=" * 78)
print("Conversación cerrada. Un contextId por agente:",
      {nombre: datos["contexto"][:8] + "…" for nombre, datos in AGENTES.items()})

TURNO 1 · usuario: Reto del taller: evaluar si conviene invertir en Falabella. ¿Qué datos y qué criterios debería revisar antes de decidir?


C:\Users\Carlos Zapata Z\Downloads\UAI - Certificado Agentic AI\Taller UAI IA\grupo-1-taller-a2a\venv-notebook\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[supervisor] -> agente_bryhan · La consulta requiere evaluar una inversión financiera y analizar datos de negocio, lo cual corresponde al área de AI Engineering aplicado a procesos empresariales y financieros.


[agente_bryhan] respondió 3863 caracteres

[respuesta de agente_bryhan]
**Conclusión:** 
Evaluar una inversión en Falabella requiere un análisis riguroso basado en datos financieros, operacionales y de mercado, estructurados bajo criterios de rentabilidad, eficiencia, riesgo y alineación con la estrategia digital y de retail financiero de la compañía. Dado que la evaluación de inversiones bursátiles y corporativas excede la especialidad de Bryhan en AI Engineering y Data Science aplicado, esta respuesta se enfoca en los datos y criterios analíticos recomendados, señalando que la decisión financiera final debe ser validada por especialistas en finanzas corporativas o gestión de inversiones.

---

### Criterios y Datos a Revisar

#### 1. Criterio de Rentabilidad y Salud Financiera
*   **Datos requeridos:** Estados financieros consolidados (Estado de Resultados, Balance General y Flujo de Caja) de los últimos periodos.
*   **Métrica clave:** 
    *   **EBITDA y márgenes operacionales:** P

C:\Users\Carlos Zapata Z\Downloads\UAI - Certificado Agentic AI\Taller UAI IA\grupo-1-taller-a2a\venv-notebook\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[supervisor] -> agente_bryhan · Es experto en AI Engineering aplicado a procesos empresariales, por lo que es el indicado para diseñar un proceso automatizado con IA y aconsejar sobre qué evitar.


[agente_bryhan] respondió 3915 caracteres

[respuesta de agente_bryhan]
**Conclusión:** 
Para automatizar el análisis financiero y estratégico de Falabella con IA de forma rigurosa, se debe diseñar un flujo basado en procesamiento de documentos financieros (reportes trimestrales y estados financieros), extracción de métricas clave mediante LLMs con técnicas de validación estricta, y estructuración de datos para comparativa. Se debe evitar delegar decisiones financieras críticas a la IA y asumir que un modelo de lenguaje puede realizar cálculos financieros complejos sin errores.

---

### Diseño Conceptual de la Solución de Automatización

Siguiendo el enfoque estructurado de Bryhan en proyectos de automatización y procesamiento documental (como el uso de Python, Azure Document Intelligence y flujos de datos en su trayectoria documentada), el proceso se diseña en cinco etapas:

1.  **Ingesta y Procesamiento Documental (Data Ingestion & Parsing):**
    *   *Componentes:* Repositorio de d

C:\Users\Carlos Zapata Z\Downloads\UAI - Certificado Agentic AI\Taller UAI IA\grupo-1-taller-a2a\venv-notebook\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[supervisor] -> agente_soluciones_comerciales_marketing · La consulta solicita un criterio estratégico y comercial sobre decisiones de inversión, lo cual se alinea directamente con las competencias de análisis de necesidades, comparación de alternativas y propuesta de soluciones comerciales del agente de Carlos Monroy.


[agente_soluciones_comerciales_marketing] respondió 1401 caracteres

[respuesta de agente_soluciones_comerciales_marketing]
Con gusto. Desde una perspectiva analítica y comercial, el criterio general para decidir cuándo conviene invertir y cuándo no se basa en la evaluación objetiva de la rentabilidad, el riesgo y la evidencia documental disponible.

**Conviene invertir cuando:**
1. **Existe un caso de negocio fundamentado:** Los datos de mercado, la observación directa y los antecedentes respaldan que el retorno esperado supera el costo de capital y los riesgos asociados.
2. **Se cuenta con respaldo documental:** Los estados financieros, proyecciones y contratos han sido verificados, distinguiendo claramente entre hechos comprobables y supuestos de mercado.
3. **Se dispone de holgura financiera:** La inversión no compromete la operatividad ni la liquidez necesaria para el funcionamiento diario.

**No conviene invertir cuando:**
1. **Los retornos son inciertos o insuficientes:** La inf

## Cierre

Se liberan los clientes A2A y el cliente HTTP. Ejecutar esta celda al terminar.

In [15]:
for datos in AGENTES.values():
    await datos["cliente"].close()
for nombre in ("cliente_a2a",):
    if nombre in globals():
        await globals()[nombre].close()
await http_a2a.aclose()
print("Conexiones cerradas.")

Conexiones cerradas.
